# RealThor v0.1 — Listing Supply Intelligence

This notebook is the starting point for Module 1. It shows RealThor running directly in a notebook before any project structure or versioning is in place.

**Use case:** Given demand and supply signals for a region, should we prioritize new listings there?

> **Note:** This notebook intentionally has the prompt inline (not externalized). The demo shows why that is a problem.

## Setup

In [ ]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Load Ciudaty Data

In [ ]:
demand = pd.read_csv("demand_signals.csv")

In [ ]:
print("Demand signals:")
display(demand)

In [ ]:
supply = pd.read_csv("supply_snapshot.csv")

In [ ]:
print("\nSupply snapshot:")
display(supply)

## RealThor v0.1

The system prompt is defined inline here. In Video 3, we will extract it to a versioned file.

In [ ]:
# NOTE: prompt is inline — this is the problem we will fix soon
SYSTEM_PROMPT = """
You are RealThor, a Listing Supply Intelligence agent for a PropTech operating in Ciudaty.

Your task is to analyze demand and supply signals for a given region and property profile.

Use the available data to recommend whether the company should prioritize bringing more
properties of that type to the platform.

Your recommendation must include:
- priority: high, medium, or low
- reasoning based on demand
- reasoning based on supply
- recommended action
- confidence: high, medium, or low

Do not make claims that are not supported by the provided data.
"""

In [ ]:
def build_context(region: str, property_type: str, bedrooms: int) -> str:
    d = demand[
        (demand["region"] == region)
        & (demand["property_type"] == property_type)
        & (demand["bedrooms"] == bedrooms)
    ]
    s = supply[
        (supply["region"] == region)
        & (supply["property_type"] == property_type)
        & (supply["bedrooms"] == bedrooms)
    ]

    if d.empty or s.empty:
        return "No data available for the requested segment."

    return (
        f"Region: {region}\n"
        f"Property: {bedrooms}-bedroom {property_type}\n\n"
        f"Demand signals:\n{d.to_string(index=False)}\n\n"
        f"Supply snapshot:\n{s.to_string(index=False)}"
    )

In [ ]:
def ask_realthor(region: str, property_type: str, bedrooms: int) -> str:
    context = build_context(region, property_type, bedrooms)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": context},
        ],
    )
    return response.choices[0].message.content

## First Recommendation

In [ ]:
recommendation = ask_realthor(region="west", property_type="apartment", bedrooms=2)
print(recommendation)

## Try Another Segment

In [ ]:
recommendation = ask_realthor(region="north", property_type="apartment", bedrooms=1)
print(recommendation)

---

## What we just built

RealThor works. But:

- The system prompt is **hardcoded inline**. If we change it, we lose the previous version.
- There is **no project structure** — the notebook and the data are in the same flat folder.
- There is **no environment pinning** — `pip install openai` could install any version.
- If a teammate runs this notebook tomorrow and gets a different answer, **we cannot explain why**.

In the next videos, we fix all of this.